In [4]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import math
import re

import sys
sys.path.append('../../..')
from mount_drive import mount_s_drive

In [5]:
mount_s_drive(subfolder='LCICM/Databases/eICU')

Password for mbranda1: ········


Mounting S drive subfolder LCICM/Databases/eICU
The following files and folders have been mounted to /home/idies/workspace/SAFE/:
  lab.csv
  respiratoryCharting.csv
  nurseCharting.csv
  patient.csv
  apacheApsVar.csv
  respiratoryCare.csv
  vitalPeriodic.csv
  carePlanEOL.csv
  nurseAssessment.csv
  infusionDrug.csv
  hospital.csv
  nurseCare.csv
  carePlanInfectiousDisease.csv
  physicalExam.csv
  carePlanGoal.csv
  admissionDx.csv
  carePlanCareProvider.csv
  medication.csv
  note.csv
  customLab.csv
  vitalAperiodic.csv
  apachePatientResult.csv
  carePlanGeneral.csv
  intakeOutput.csv
  admissionDrug.csv
  pastHistory.csv
  treatment.csv
  apachePredVar.csv
  allergy.csv
  diagnosis.csv
  microLab.csv


mkdir: cannot create directory '/home/idies/workspace/SAFE/': File exists


In [110]:
myHours = 60 * 6

In [111]:
database_folder = '/home/idies/workspace/SAFE/'

### Patients 

In [112]:
patients_df = pd.read_csv(database_folder+'patient.csv')

In [113]:
myIds = pd.read_csv('../../../Isalis/eICU/eICU_patientunitstayids.txt', header = None)
myIds

,0
0,142723.0
1,145204.0
2,145205.0
3,145830.0
4,146619.0
...,...
3400,3352445.0
3401,3352512.0
3402,3352727.0
3403,3353194.0


In [114]:
patients_df = patients_df[patients_df['patientunitstayid'].isin(myIds[0])]

In [115]:
myPredictorsDf = patients_df[['patientunitstayid', 'gender', 'age', 'apacheadmissiondx', 'admissionheight', 'hospitaladmittime24', 'hospitaladmitsource', 'admissionweight']]

### Treatments

In [116]:
def getOneHotConditions(aDf, aColumn, aPrefix):
    aDf['conditions'] = aDf[aColumn].str.split('|')
    one_hot = aDf['conditions'].explode().str.get_dummies().groupby(level=0).sum()
    one_hot = one_hot.add_prefix(aPrefix)
    aDf = aDf.drop(columns=['conditions']).join(one_hot)
    return aDf, one_hot

In [117]:
treatment_df = pd.read_csv(database_folder+'treatment.csv')
treatment_df = treatment_df[treatment_df['patientunitstayid'].isin(myIds[0])]

In [118]:
treatment_df, one_hot = getOneHotConditions(treatment_df, 'treatmentstring', 'treatment_')

In [119]:
filtered_df = treatment_df[(treatment_df.treatmentoffset < myHours)].groupby('patientunitstayid').agg('sum').reset_index()
filtered_df_hyp = treatment_df.groupby('patientunitstayid').agg('sum').reset_index()

/tmp/ipykernel_233/1726350892.py:1: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  filtered_df = treatment_df[(treatment_df.treatmentoffset < myHours)].groupby('patientunitstayid').agg('sum').reset_index()
/tmp/ipykernel_233/1726350892.py:2: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  filtered_df_hyp = treatment_df.groupby('patientunitstayid').agg('sum').reset_index()


In [120]:
filtered_df[one_hot.columns] = (filtered_df[one_hot.columns] != 0).astype(int)
filtered_df_hyp['hyp'] = (filtered_df_hyp.treatment_hypothermia != 0).astype(int)
filtered_df_hyp.hyp.sum()

603

In [121]:
myPredictorsDf1 = myPredictorsDf.merge(filtered_df[['patientunitstayid'] + list(one_hot.columns)], on=['patientunitstayid'], how='left')
myPredictorsDf1 = myPredictorsDf1.merge(filtered_df_hyp[['patientunitstayid', 'hyp']], on=['patientunitstayid'], how='left')
myPredictorsDf1['treatment_hypothermia'] = myPredictorsDf1.hyp
myPredictorsDf1.drop(columns=['hyp'], inplace = True)

In [122]:
myPredictorsDf = myPredictorsDf1

### Diagnosis

In [123]:
diagnosis_df = pd.read_csv(database_folder+'diagnosis.csv')
diagnosis_df = diagnosis_df[diagnosis_df['patientunitstayid'].isin(myIds[0])]

In [124]:
diagnosis_df, one_hot = getOneHotConditions(diagnosis_df, 'diagnosisstring', 'diagnosis_')

In [125]:
filtered_df = diagnosis_df[(diagnosis_df.diagnosisoffset < myHours)].groupby('patientunitstayid').agg('sum').reset_index()
filtered_df[one_hot.columns] = (filtered_df[one_hot.columns] != 0).astype(int)

/tmp/ipykernel_233/3375727360.py:1: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  filtered_df = diagnosis_df[(diagnosis_df.diagnosisoffset < myHours)].groupby('patientunitstayid').agg('sum').reset_index()


In [126]:
myPredictorsDf1 = myPredictorsDf.merge(filtered_df[['patientunitstayid'] + list(one_hot.columns)], on=['patientunitstayid'], how='left')

In [127]:
myPredictorsDf = myPredictorsDf1

### Nurse Charting

In [128]:
file_path = database_folder+'nurseCharting.csv'
chunk_size = 1e6

df_chunks = []

num_chunks = 0
for chunk in pd.read_csv(file_path,chunksize=chunk_size):
    filtered_chunk = chunk[chunk['patientunitstayid'].isin(myIds[0])]
    df_chunks.append(filtered_chunk)
    if num_chunks % 10 == 0:
        print(f'Chunk {num_chunks} Processed.')
    num_chunks += 1
    del chunk

nurse_charting_df = pd.concat(df_chunks, ignore_index=True)
print('Processing finished.')

Chunk 0 Processed.
Chunk 10 Processed.
Chunk 20 Processed.
Chunk 30 Processed.
Chunk 40 Processed.
Chunk 50 Processed.
Chunk 60 Processed.
Chunk 70 Processed.
Chunk 80 Processed.
Chunk 90 Processed.
Chunk 100 Processed.
Chunk 110 Processed.
Chunk 120 Processed.
Chunk 130 Processed.
Chunk 140 Processed.
Chunk 150 Processed.
Processing finished.


### Hypothermmia Analysis

In [129]:
myDfTempC = nurse_charting_df[(nurse_charting_df.nursingchartcelltypevalname == 'Temperature (C)')]
myDfTempC.nursingchartvalue = myDfTempC.nursingchartvalue.astype(float)
myDfTempC.sort_values(by=['patientunitstayid', 'nursingchartentryoffset'], inplace=True)
# myDfTempC['TimeDiff'] = myDfTempC['nursingchartoffset'].diff()
# myDfTempC['TempTimesTime'] = myDfTempC['TimeDiff'] * myDfTempC['nursingchartvalue'].shift(1)
# myDfTempC['RollingTemp'] = myDfTempC['TempTimesTime'].rolling(window=2).sum()
# myDfTempC['RollingTime'] = myDfTempC['TimeDiff'].rolling(window=2).sum()
# myDfTempC['AvgTemp'] = myDfTempC['RollingTemp'] / myDfTempC['RollingTime']
# myDfTempC[['TempTimesTime', 'TimeDiff', 'nursingchartvalue', 'RollingTemp', 'RollingTime', 'AvgTemp']]
myDfTempC2 = myDfTempC[(myDfTempC.nursingchartentryoffset < 2880)  \
        & (myDfTempC.nursingchartentryoffset > 0)
        & (myDfTempC.nursingchartvalue < 36)]
myDfTempC2['nursingchartentryoffset2'] = myDfTempC2['nursingchartentryoffset']
myHyp = myDfTempC2.groupby('patientunitstayid').agg({'patientunitstayid':'count', 'nursingchartvalue': 'min', 'nursingchartentryoffset':'min', 'nursingchartentryoffset2':'max'})
myHyp['Hyp'] = (myHyp['patientunitstayid'] > 12).astype(int) \
                & (myHyp['nursingchartvalue'] > 25).astype(int) \
                & (myHyp['nursingchartentryoffset2'] - myHyp['nursingchartentryoffset'] > 720).astype(int)
myHyp[(myHyp['Hyp'] == 1)] 

/tmp/ipykernel_233/3276566090.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  myDfTempC.nursingchartvalue = myDfTempC.nursingchartvalue.astype(float)
/tmp/ipykernel_233/3276566090.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  myDfTempC.sort_values(by=['patientunitstayid', 'nursingchartentryoffset'], inplace=True)
/tmp/ipykernel_233/3276566090.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydat

,patientunitstayid,nursingchartvalue,nursingchartentryoffset,nursingchartentryoffset2,Hyp
patientunitstayid,,,,,
153418,116,33.0,11,2336,1
161900,32,33.1,351,1971,1
162897,164,27.8,498,2878,1
165286,104,32.4,468,2778,1
166054,135,33.3,259,2584,1
...,...,...,...,...,...
3352203,68,32.6,151,2101,1
3352445,59,33.0,71,2141,1
3352512,105,32.2,22,2512,1


In [130]:
myPredictorsDf['Hypothermia'] = 0
myPredictorsDf.loc[myPredictorsDf.patientunitstayid.isin(myHyp[myHyp['Hyp'] == 1].index), 'Hypothermia'] = 1
myPredictorsDf[['Hypothermia', 'patientunitstayid']]
myPredictorsDf.Hypothermia.sum()
# myPredictorsDf.drop(columns='Hypothermia', inplace=True)

780

In [131]:
myPredictorsDf['both_hypothermia'] = ((myPredictorsDf.treatment_hypothermia == 1) | (myPredictorsDf.Hypothermia == 1)).astype(int)

In [132]:
(myDfTempC[(myDfTempC.patientunitstayid ==3243869) & (myDfTempC.nursingchartentryoffset < 500)])

,nursingchartid,patientunitstayid,nursingchartoffset,nursingchartentryoffset,nursingchartcelltypecat,nursingchartcelltypevallabel,nursingchartcelltypevalname,nursingchartvalue
5499450,2241588686,3243869,19,19,Vital Signs,Temperature,Temperature (C),35.0
5499898,2242081401,3243869,34,34,Vital Signs,Temperature,Temperature (C),35.2
5499691,2218814493,3243869,49,49,Vital Signs,Temperature,Temperature (C),35.3
5498361,2230898102,3243869,64,64,Vital Signs,Temperature,Temperature (C),35.4
5498196,2225685817,3243869,79,79,Vital Signs,Temperature,Temperature (C),35.5
5499959,2272143813,3243869,94,94,Vital Signs,Temperature,Temperature (C),35.5
5498201,2210630692,3243869,109,109,Vital Signs,Temperature,Temperature (C),35.4
5499365,2272237371,3243869,124,124,Vital Signs,Temperature,Temperature (C),35.4
5499745,2258855696,3243869,139,139,Vital Signs,Temperature,Temperature (C),35.4
5498305,2260942118,3243869,154,154,Vital Signs,Temperature,Temperature (C),35.4


### Getting Features 

In [133]:
def getFeaturesFromDf(aDf, aTimeColumn, aTypeColumn, aValueColumn):
    aDf['num_values'] = pd.to_numeric(aDf[aValueColumn], errors='coerce')
    myMasterGroupDf = aDf[(aDf[aTimeColumn] < myHours)].groupby(['patientunitstayid', aTypeColumn])
    myMasterGroupBegin = aDf.loc[myMasterGroupDf[aTimeColumn].idxmin()]
    myMasterGroupEnd = aDf.loc[myMasterGroupDf[aTimeColumn].idxmax()]
    myMasterGroupAgg = myMasterGroupDf.agg({'mean', 'min', 'max'})
    myMasterGroupAgg = myMasterGroupAgg.num_values.reset_index()
    return myMasterGroupDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg

In [134]:
def mergeFeaturesInDf(aDfToMerge, aBegin, aEnd, aAgg, aPrefix, aTypeColumn, aValueColumn):
    print('Running Begin Columns')
    total = int(aBegin[aTypeColumn].nunique() / 10)
    myPredictorsDf1 = aDfToMerge
    i = 0
    print('progress: ', end="")
    for value in aBegin[aTypeColumn].unique():
        myDf = aBegin[aBegin[aTypeColumn] == value]
        myPredictorsDf1 = myPredictorsDf1.merge(myDf[['patientunitstayid', aValueColumn]], on='patientunitstayid', how='left')
        myPredictorsDf1[aPrefix + '_first_' + value] = myPredictorsDf1[aValueColumn]
        myPredictorsDf1.drop(columns=[aValueColumn], inplace=True)
        if (i % total == 0):
            print('#', end="")
        i+= 1
    print()
    print('Running End Columns')
    total = int(aEnd[aTypeColumn].nunique() / 10)
    i = 0
    print('progress: ', end="")
    for value in aEnd[aTypeColumn].unique():
        myDf = aEnd[aEnd[aTypeColumn] == value]
        myPredictorsDf1 = myPredictorsDf1.merge(myDf[['patientunitstayid', aValueColumn]], on='patientunitstayid', how='left')
        myPredictorsDf1[aPrefix + '_last_' + value] = myPredictorsDf1[aValueColumn]
        myPredictorsDf1.drop(columns=[aValueColumn], inplace=True)
        if (i % total == 0):
            print('#', end="")
        i+= 1
    print()
    print('Running Agg Columns')
    total = int(aEnd[aTypeColumn].nunique() / 10)
    i = 0
    print('progress: ', end="")
    for value in aAgg[aTypeColumn].unique():
        myDf = aAgg[aAgg[aTypeColumn] == value]
        myPredictorsDf1 = myPredictorsDf1.merge(myDf[['patientunitstayid', 'max', 'min', 'mean']], on='patientunitstayid', how='left')
        myPredictorsDf1[aPrefix + '_max_' + value] = myPredictorsDf1['max']
        myPredictorsDf1[aPrefix + '_min_' + value] = myPredictorsDf1['min']
        myPredictorsDf1[aPrefix + '_mean_' + value] = myPredictorsDf1['mean']
        myPredictorsDf1.drop(columns=['max', 'min', 'mean'], inplace=True)
        if (i % total == 0):
            print('#', end="")
        i+= 1
    return myPredictorsDf1

In [135]:
myMasterGroupDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg = getFeaturesFromDf(nurse_charting_df, 'nursingchartoffset', 'nursingchartcelltypevalname', 'nursingchartvalue')

/tmp/ipykernel_233/3054475834.py:6: FutureWarning: ['nursingchartcelltypecat', 'nursingchartcelltypevallabel', 'nursingchartvalue'] did not aggregate successfully. If any error is raised this will raise in a future version of pandas. Drop these columns/ops to avoid this warning.
  myMasterGroupAgg = myMasterGroupDf.agg({'mean', 'min', 'max'})


In [136]:
myPredictorsDf1 = mergeFeaturesInDf(myPredictorsDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg, 'nurse', 'nursingchartcelltypevalname', 'nursingchartvalue')

Running Begin Columns
progress: #############
Running End Columns
progress: ############

/tmp/ipykernel_233/2942944636.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  myPredictorsDf1[aPrefix + '_last_' + value] = myPredictorsDf1[aValueColumn]
/tmp/ipykernel_233/2942944636.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  myPredictorsDf1[aPrefix + '_last_' + value] = myPredictorsDf1[aValueColumn]
/tmp/ipykernel_233/2942944636.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at 

#
Running Agg Columns
progress: #

/tmp/ipykernel_233/2942944636.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  myPredictorsDf1[aPrefix + '_max_' + value] = myPredictorsDf1['max']
/tmp/ipykernel_233/2942944636.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  myPredictorsDf1[aPrefix + '_min_' + value] = myPredictorsDf1['min']
/tmp/ipykernel_233/2942944636.py:38: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.co

#

...

/tmp/ipykernel_233/2942944636.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  myPredictorsDf1[aPrefix + '_max_' + value] = myPredictorsDf1['max']
/tmp/ipykernel_233/2942944636.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  myPredictorsDf1[aPrefix + '_min_' + value] = myPredictorsDf1['min']
/tmp/ipykernel_233/2942944636.py:38: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.co

#

/tmp/ipykernel_233/2942944636.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  myPredictorsDf1[aPrefix + '_max_' + value] = myPredictorsDf1['max']
/tmp/ipykernel_233/2942944636.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  myPredictorsDf1[aPrefix + '_min_' + value] = myPredictorsDf1['min']
/tmp/ipykernel_233/2942944636.py:38: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.co

##

/tmp/ipykernel_233/2942944636.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  myPredictorsDf1[aPrefix + '_max_' + value] = myPredictorsDf1['max']
/tmp/ipykernel_233/2942944636.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  myPredictorsDf1[aPrefix + '_min_' + value] = myPredictorsDf1['min']
/tmp/ipykernel_233/2942944636.py:38: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.co

In [137]:
print(list(myPredictorsDf.columns))

['patientunitstayid', 'gender', 'age', 'apacheadmissiondx', 'admissionheight', 'hospitaladmittime24', 'hospitaladmitsource', 'admissionweight', 'treatment_10-15 cm H2O', 'treatment_25-30%', 'treatment_30-40%', 'treatment_40-50%', 'treatment_5-10 cm H2O', 'treatment_50-60%', 'treatment_500-1000 ml', 'treatment_60-70%', 'treatment_70-80%', 'treatment_80-90%', 'treatment_> 15 cm H2O', 'treatment_>250 cc/hr', 'treatment_>90%', 'treatment_ACE inhibitor', 'treatment_AFB', 'treatment_AICD placement', 'treatment_Amicar (aminocaproic acid)', 'treatment_BAL/PBS', 'treatment_C A V H D', 'treatment_C V V H', 'treatment_C V V H D', 'treatment_C. difficile toxin', 'treatment_CABG', 'treatment_CPAP/PEEP therapy', 'treatment_CSF', 'treatment_CSF drainage via ventriculostomy', 'treatment_CT scan', 'treatment_Cardiac surgery consultation', 'treatment_Cardiology consultation', 'treatment_D10W', "treatment_D5 Lactated Ringer's", 'treatment_D5 half-normal saline', 'treatment_D50', 'treatment_D5NS', 'treatm

In [138]:
myPredictorsDf = myPredictorsDf1

In [139]:
gcs_df = nurse_charting_df[nurse_charting_df['nursingchartcelltypevalname']=='GCS Total']

In [140]:
motor_gcs_df = nurse_charting_df[nurse_charting_df.nursingchartcelltypevalname == 'Motor']

In [141]:
myGcsDf = gcs_df.sort_values(by=['patientunitstayid', "nursingchartoffset"])
myGcsDf['nursingchartoffset2'] = myGcsDf.nursingchartoffset
myGcsDf2 = myGcsDf.groupby('patientunitstayid').agg({'nursingchartoffset':'min', 'nursingchartoffset2': 'max'})
myGcsDfMin = myGcsDf.merge(myGcsDf2, on=['patientunitstayid', 'nursingchartoffset'], how='inner')
myGcsDfMax = myGcsDf.merge(myGcsDf2, on=['patientunitstayid', 'nursingchartoffset2'], how='inner')

In [142]:
myMGcsDf = motor_gcs_df.sort_values(by=['patientunitstayid', 'nursingchartoffset'])
myMGcsDf['nursingchartoffset2'] = myMGcsDf.nursingchartoffset
myMGcsDf2 = myMGcsDf.groupby('patientunitstayid').agg({'nursingchartoffset':'min', 'nursingchartoffset2': 'max'})
myMGcsDfMin = myMGcsDf.merge(myMGcsDf2, on=['patientunitstayid', 'nursingchartoffset'], how='inner')
myMGcsDfMax = myMGcsDf.merge(myMGcsDf2, on=['patientunitstayid', 'nursingchartoffset2'], how='inner')

In [143]:
myMGcsDfMax

,nursingchartid,patientunitstayid,nursingchartoffset_x,nursingchartentryoffset,nursingchartcelltypecat,nursingchartcelltypevallabel,nursingchartcelltypevalname,nursingchartvalue,num_values,nursingchartoffset2,nursingchartoffset_y
0,138827211,158800,8,89,Scores,Glasgow coma score,Motor,1,1.0,8,8
1,174909972,168071,10585,10599,Scores,Glasgow coma score,Motor,6,6.0,10585,10585
2,138825186,168324,108,216,Scores,Glasgow coma score,Motor,1,1.0,108,108
3,138769012,192236,35510,35526,Scores,Glasgow coma score,Motor,4,4.0,35510,35510
4,156847734,193911,10502,10577,Scores,Glasgow coma score,Motor,5,5.0,10502,10502
...,...,...,...,...,...,...,...,...,...,...,...
2929,2287942733,3351806,4566,4566,Scores,Glasgow coma score,Motor,1,1.0,4566,3132
2930,2291882680,3351912,130390,130390,Scores,Glasgow coma score,Motor,5,5.0,130390,2965
2931,2284258361,3352159,62,62,Scores,Glasgow coma score,Motor,1,1.0,62,62
2932,2311451473,3352727,3716,3716,Scores,Glasgow coma score,Motor,6,6.0,3716,2516


In [144]:
myPredictorsDf = myPredictorsDf.merge(myGcsDfMin[['patientunitstayid', 'nursingchartentryoffset', 'nursingchartvalue']], on='patientunitstayid')
myPredictorsDf.rename(columns={'nursingchartentryoffset': 'FirstGCSTime', 'nursingchartvalue': 'FirstGCS'}, inplace=True)
myPredictorsDf =myPredictorsDf.merge(myGcsDfMax[['patientunitstayid', 'nursingchartentryoffset', 'nursingchartvalue']], on='patientunitstayid')
myPredictorsDf.rename(columns={'nursingchartentryoffset': 'LastGCSTime', 'nursingchartvalue': 'LastGCS'}, inplace=True)

In [145]:
myPredictorsDf = myPredictorsDf.merge(myMGcsDfMin[['patientunitstayid', 'nursingchartentryoffset', 'nursingchartvalue']], on='patientunitstayid', how='outer')
myPredictorsDf.rename(columns={'nursingchartentryoffset': 'FirstMGCSTime', 'nursingchartvalue': 'FirstMGCS'}, inplace=True)
myPredictorsDf =myPredictorsDf.merge(myMGcsDfMax[['patientunitstayid', 'nursingchartentryoffset', 'nursingchartvalue']], on='patientunitstayid', how='outer')
myPredictorsDf.rename(columns={'nursingchartentryoffset': 'LastMGCSTime', 'nursingchartvalue': 'LastMGCS'}, inplace=True)

### Labs

In [146]:
file_path = database_folder+'lab.csv'
chunk_size = 1e6

# patient_unit_stay_ids = ca_patients_df['patientunitstayid']
df_chunks = []

num_chunks = 0
for chunk in pd.read_csv(file_path,chunksize=chunk_size):
    filtered_chunk = chunk[chunk['patientunitstayid'].isin(myIds[0])]
    df_chunks.append(filtered_chunk)
    if num_chunks % 10 == 0:
        print(f'Chunk {num_chunks} Processed.')
    num_chunks += 1
    del chunk

myLabs = pd.concat(df_chunks, ignore_index=True)
print('Processing finished.')

Chunk 0 Processed.
Chunk 10 Processed.
Chunk 20 Processed.
Chunk 30 Processed.
Processing finished.


In [147]:
myLabs

,labid,patientunitstayid,labresultoffset,labtypeid,labname,labresult,labresulttext,labmeasurenamesystem,labmeasurenameinterface,labresultrevisedoffset
0,45551268,142723,-21,1,magnesium,1.700,1.7,mg/dL,mg/dL,18
1,51593737,142723,-21,3,MCH,27.300,27.3,pg,pg,-11
2,47238164,142723,105,1,lactate,5.700,5.7,mmol/L,mmol/L,133
3,47168398,142723,152,1,albumin,1.300,1.3,g/dL,g/dL,199
4,49725004,142723,891,2,Vancomycin - random,9.600,9.6,mcg/mL,mcg/mL,946
...,...,...,...,...,...,...,...,...,...,...
1293181,826720664,3353251,4049,4,bedside glucose,129.000,129,mg/dL,mg/dL,4049
1293182,822488872,3353251,1849,1,BUN,32.000,32,mg/dL,mg/dL,1894
1293183,829194475,3353251,310,7,pH,7.194,7.194,NaN,NaN,310
1293184,822355171,3353251,409,1,potassium,4.400,4.4,mmol/L,mmol/L,451


In [148]:
myMasterGroupDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg = getFeaturesFromDf(myLabs, 'labresultoffset', 'labname', 'labresult')

/tmp/ipykernel_233/3054475834.py:6: FutureWarning: ['labresulttext', 'labmeasurenamesystem', 'labmeasurenameinterface'] did not aggregate successfully. If any error is raised this will raise in a future version of pandas. Drop these columns/ops to avoid this warning.
  myMasterGroupAgg = myMasterGroupDf.agg({'mean', 'min', 'max'})


In [149]:
myPredictorsDf1 = mergeFeaturesInDf(myPredictorsDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg, 'lab', 'labname', 'labresult')

Running Begin Columns
progress: #

/tmp/ipykernel_233/2942944636.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  myPredictorsDf1[aPrefix + '_first_' + value] = myPredictorsDf1[aValueColumn]
/tmp/ipykernel_233/2942944636.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  myPredictorsDf1[aPrefix + '_first_' + value] = myPredictorsDf1[aValueColumn]
/tmp/ipykernel_233/2942944636.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns a

#

/tmp/ipykernel_233/2942944636.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  myPredictorsDf1[aPrefix + '_first_' + value] = myPredictorsDf1[aValueColumn]
/tmp/ipykernel_233/2942944636.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  myPredictorsDf1[aPrefix + '_first_' + value] = myPredictorsDf1[aValueColumn]
/tmp/ipykernel_233/2942944636.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns a

#

...

/tmp/ipykernel_233/2942944636.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  myPredictorsDf1[aPrefix + '_max_' + value] = myPredictorsDf1['max']
/tmp/ipykernel_233/2942944636.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  myPredictorsDf1[aPrefix + '_min_' + value] = myPredictorsDf1['min']
/tmp/ipykernel_233/2942944636.py:38: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.co

#

/tmp/ipykernel_233/2942944636.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  myPredictorsDf1[aPrefix + '_max_' + value] = myPredictorsDf1['max']
/tmp/ipykernel_233/2942944636.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  myPredictorsDf1[aPrefix + '_min_' + value] = myPredictorsDf1['min']
/tmp/ipykernel_233/2942944636.py:38: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.co

#

/tmp/ipykernel_233/2942944636.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  myPredictorsDf1[aPrefix + '_max_' + value] = myPredictorsDf1['max']
/tmp/ipykernel_233/2942944636.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  myPredictorsDf1[aPrefix + '_min_' + value] = myPredictorsDf1['min']
/tmp/ipykernel_233/2942944636.py:38: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.co

In [150]:
[x for x in myPredictorsDf.columns if 'first' in x]

['treatment_first generation cephalosporin',
 'diagnosis_first degree',
 'nurse_first_GCS Total',
 'nurse_first_Heart Rate',
 'nurse_first_Non-Invasive BP Diastolic',
 'nurse_first_Non-Invasive BP Mean',
 'nurse_first_Non-Invasive BP Systolic',
 'nurse_first_O2 Saturation',
 'nurse_first_Pain Score',
 'nurse_first_Respiratory Rate',
 'nurse_first_Temperature (C)',
 'nurse_first_Temperature (F)',
 'nurse_first_Temperature Location',
 'nurse_first_Value',
 'nurse_first_Invasive BP Diastolic',
 'nurse_first_Invasive BP Systolic',
 'nurse_first_CI',
 'nurse_first_PVR',
 'nurse_first_Bedside Glucose',
 'nurse_first_Eyes',
 'nurse_first_Motor',
 'nurse_first_Verbal',
 'nurse_first_Invasive BP Mean',
 'nurse_first_O2 L/%',
 'nurse_first_Sedation Scale',
 'nurse_first_Sedation Score',
 'nurse_first_Delirium Scale',
 'nurse_first_Delirium Score',
 'nurse_first_End Tidal CO2',
 'nurse_first_CPP',
 'nurse_first_ICP',
 'nurse_first_CVP',
 'nurse_first_O2 Admin Device',
 'nurse_first_CO',
 'nurse_f

In [151]:
myPredictorsDf = myPredictorsDf1

In [152]:
myPredictorsDf = myPredictorsDf.merge(patients_df[['patientunitstayid', 'hospitaldischargestatus']], on=['patientunitstayid'])

In [153]:
myPredictorsDf['DeathAtDischarge'] = myPredictorsDf.hospitaldischargestatus == 'Expired'
myPredictorsDf['DeathAtDischarge'] = myPredictorsDf['DeathAtDischarge'].astype(int)

/tmp/ipykernel_233/3215504514.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  myPredictorsDf['DeathAtDischarge'] = myPredictorsDf.hospitaldischargestatus == 'Expired'


In [155]:
myPredictorsDf['LastGCS15'] = (myPredictorsDf['LastGCS'] == '15').astype(int)

In [156]:
myPredictorsDf.loc[myPredictorsDf.age == '> 89', 'age'] = 89

In [157]:
myPredictorsDf.age = myPredictorsDf.age.astype(int)

In [158]:
myPredictorsDf.to_csv('eICUPredictorsDiag.csv', index=False)

/home/idies/miniconda3/lib/python3.9/site-packages/pandas/core/internals/blocks.py:2323: RuntimeWarning: invalid value encountered in cast
  values = values.astype(str)
